# Phase 3: Data Cleaning & Preprocessing
In this notebook, we prepare the raw data for machine learning by handling categorical variables, scaling numerical features, and creating a clean dataset.

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# Load the dataset
df = pd.read_csv('../data/processed/customer_details_processed.csv')
print("Original shape:", df.shape)

Original shape: (3900, 18)


In [16]:
# 1. Handle Missing Values and Duplicates
print("Missing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

# Drop duplicates if any
df = df.drop_duplicates()

Missing values:
 Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

Duplicates: 0


In [17]:
# 2. Drop Identifier columns (Not useful for ML)
if 'Customer ID' in df.columns:
    df = df.drop(['Customer ID'], axis=1)

In [18]:
# 3. Binary Encoding for Yes/No columns
binary_cols = ['Subscription Status', 'Discount Applied', 'Promo Code Used']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

display(df[binary_cols].head())

,Subscription Status,Discount Applied,Promo Code Used
0,1,1,1
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1


In [19]:
# 4. Ordinal Encoding for Size and Frequency
# Map sizes to numbers (S=1, M=2, L=3, XL=4)
size_mapping = {'S': 1, 'M': 2, 'L': 3, 'XL': 4}
df['Size'] = df['Size'].map(size_mapping)

# Map frequencies to numbers (Annually=1, Weekly=5)
freq_mapping = {
    'Annually': 1, 
    'Every 3 Months': 2, 'Quarterly': 2,
    'Monthly': 3,
    'Bi-Weekly': 4, 'Fortnightly': 4,
    'Weekly': 5
}
df['Frequency of Purchases'] = df['Frequency of Purchases'].map(freq_mapping)

In [20]:
# 5. One-Hot Encoding for remaining categorical features
categorical_cols = ['Gender', 'Category', 'Season', 'Shipping Type', 'Payment Method']

# High-cardinality columns (like Location, Item Purchased, Color) are dropped 
# for now to avoid creating hundreds of columns in our baseline model.
df = df.drop(['Location', 'Item Purchased', 'Color'], axis=1)

# Get dummy variables (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", df_encoded.shape)

Shape after encoding: (3900, 26)


In [21]:
# 6. Scale Numerical Features
# We use StandardScaler so all numerical inputs have a mean of 0 and variance of 1
scaler = StandardScaler()
num_cols = ['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

display(df_encoded.head())

,Age,Purchase Amount (USD),Size,Review Rating,Subscription Status,Discount Applied,Promo Code Used,Previous Purchases,Frequency of Purchases,Gender_Male,...,Shipping Type_Express,Shipping Type_Free Shipping,Shipping Type_Next Day Air,Shipping Type_Standard,Shipping Type_Store Pickup,Payment Method_Cash,Payment Method_Credit Card,Payment Method_Debit Card,Payment Method_PayPal,Payment Method_Venmo
0,0.718913,-0.285629,3,-0.907584,1,1,1,-0.785831,4,True,...,True,False,False,False,False,False,False,False,False,True
1,-1.648629,0.178852,3,-0.907584,1,1,1,-1.616552,4,True,...,True,False,False,False,False,True,False,False,False,False
2,0.390088,0.558882,1,-0.907584,1,1,1,-0.162789,5,True,...,False,True,False,False,False,False,True,False,False,False
3,-1.517099,1.276716,2,-0.349027,1,1,1,1.637107,5,True,...,False,False,True,False,False,False,False,False,True,False
4,0.061263,-0.454531,2,-1.466141,1,1,1,0.391025,1,True,...,False,True,False,False,False,False,False,False,True,False


In [22]:
# 7. Save Processed Data
# Create directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)
processed_path = '../data/processed/customer_details_processed.csv'

df_encoded.to_csv(processed_path, index=False)
print(f"Cleaned and encoded data saved to {processed_path}")

Cleaned and encoded data saved to ../data/processed/customer_details_processed.csv
